In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import RandomizedSearchCV



In [2]:
house_data = pd.read_csv('Housing.csv')


In [3]:
house_data.head()


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
X = house_data.drop(columns=["price"])
Y = house_data["price"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=7, test_size=0.2)

In [20]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PowerTransformer, RobustScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer

# Encode categorical variables
cat_columns = X_train.select_dtypes(include=['object']).columns  
num_columns = X_train.select_dtypes(include=['int64', 'float64']).columns

# skewness = X_train[num_columns].skew()
# skewness
# skew_columns = skewness[abs(skewness) > 0.5].index.tolist()
# unskewed_columns = skewness[abs(skewness) <= 0.5].index.tolist()


num_pipeline = make_pipeline(
    (PowerTransformer(method="yeo-johnson")),
    (RobustScaler()),
)

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder())
])

multi_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


In [42]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [30]:
# Binary categorical columns
binary_cols = [
    "mainroad", "guestroom", "basement",
    "hotwaterheating", "airconditioning", "prefarea"
]

# Multi-category column
multi_cols = ["furnishingstatus"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_columns),
        ("binary", binary_pipeline, binary_cols),
        ("multi", multi_pipeline, multi_cols)
    ],
    remainder="drop"
)

In [44]:
model_pipeline = Pipeline(
    [("preprocessor", preprocessor), ("regressor", DummyRegressor())]
)


param_grid = [

    # Linear Regression
    {
        "regressor": [LinearRegression()],
        "regressor__fit_intercept": [True, False],
    },

    # Ridge
    {
        "regressor": [Ridge()],
        "regressor__alpha": [0.1, 1.0, 10.0],
        "regressor__solver": ["auto", "svd", "cholesky"],
    },

    # Lasso
    {
        "regressor": [Lasso()],
        "regressor__alpha": [0.01, 0.1, 1.0],
        "regressor__max_iter": [1000, 5000],
        "regressor__tol": [0.0001, 0.001],
    },

    # Random Forest
    {
        "regressor": [RandomForestRegressor()],
        "regressor__n_estimators": [100, 200],
        "regressor__max_depth": [5, 10, None],
        "regressor__min_samples_split": [2, 5],
        "regressor__min_samples_leaf": [1, 2],
        "regressor__max_features": ["sqrt", "log2"],
    },

    # SVR
    {
        "regressor": [SVR()],
        "regressor__C": [0.1, 1.0, 10.0],
        "regressor__kernel": ["linear", "rbf"],
        "regressor__gamma": ["scale", "auto"],
        "regressor__epsilon": [0.1, 0.2],
    },

    # 🔥 Gradient Boosting
    {
        "regressor": [GradientBoostingRegressor()],
        "regressor__n_estimators": [100, 200],
        "regressor__learning_rate": [0.01, 0.1],
        "regressor__max_depth": [3, 5],
        "regressor__subsample": [0.8, 1.0],
    },

    # 🔥🔥 XGBoost
    {
        "regressor": [XGBRegressor(objective="reg:squarederror", verbosity=0)],
        "regressor__n_estimators": [100, 300],
        "regressor__learning_rate": [0.01, 0.1],
        "regressor__max_depth": [3, 5, 7],
        "regressor__subsample": [0.8, 1.0],
        "regressor__colsample_bytree": [0.8, 1.0],
    },
]

In [45]:
grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train, Y_train)

# Best model and parameters
print("Best Score:", -grid_search.best_score_)
print("Best Model:", grid_search.best_estimator_.named_steps['regressor'])
print("Best Params:", grid_search.best_params_)

Fitting 5 folds for each of 159 candidates, totalling 795 fits


Best Score: 1227430324363.3772
Best Model: GradientBoostingRegressor(subsample=0.8)
Best Params: {'regressor': GradientBoostingRegressor(), 'regressor__learning_rate': 0.1, 'regressor__max_depth': 3, 'regressor__n_estimators': 100, 'regressor__subsample': 0.8}


In [48]:
print(Y_test[0:2])
print(grid_search.predict(X_test[0:2]))

542    1750000
70     6790000
Name: price, dtype: int64
[2813048.25473214 5738483.85517143]


In [49]:
Y_test.min(), Y_test.max()

(1750000, 12215000)

In [51]:
y_mean = Y_train.mean()
rmse = np.sqrt(1227430324363.3772)

print("Average Price:", y_mean)
print("RMSE:", rmse)
print("Error %:", rmse / y_mean * 100)

Average Price: 4809135.183486238
RMSE: 1107894.5456871684
Error %: 23.03729264029616
